# What makes the toxic reviews toxic? Evidence phrases and types

For every review that is still toxic after the cheating re-check (1,669), Gemini points at up to 3 **exact words or phrases** that make it toxic and gives each one a **type**. Then we count.

**Types** (shown with plain names in the tables) follow common toxicity schemes: Perspective API attributes (insult, profanity, identity attack, threat, sexually explicit), Risch and Krestel (2020: profanity, insults, threats, identity hate, otherwise toxic), and the game-toxicity literature (griefing, trolling, toxic community; Beres et al. 2021; Fair Play Alliance and ADL *Disruption and Harms in Online Gaming Framework*, 2020). For identity attacks Gemini also says which group (ethnicity or nationality, gender, and so on). Each piece of evidence also has a **target**: teammate, opponent, players in general, a nationality, the developer.

**Safety check.** A quote must appear word for word in the review. `quote_in_review` is `False` when it does not, so invented quotes are caught and can be dropped.

**Caveats.** The evidence is Gemini's own explanation, not proof of why it decided. Counts describe Gemini's labels, not player behavior. A phrase that points at a nationality is a statement in the review, not a fact about that nationality.

Reads `data/gemini_harm_recheck_*.csv` (read only) and writes NEW files into the folder `data/toxic_evidence/`: `gemini_toxic_evidence_*.jsonl` and `.csv`, and the spot-check file.

**Run All twice:** first with `SAMPLE = 50`, then `SAMPLE = None` for all.

## 1. Setup

In [23]:
import os, sys, json, importlib
import pandas as pd
from dotenv import load_dotenv
from google import genai

sys.path.insert(0, "../scripts")
import toxic_evidence
importlib.reload(toxic_evidence)

load_dotenv("../.env")
if os.environ.get("USE_VERTEX") == "1":
    client = genai.Client(
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
    )
else:
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 2. Choose the reviews (the one thing to change)

In [24]:
SAMPLE = None   # <-- test first with 50 toxic reviews. Then set to None and Run All again.

GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0
MODEL_TAG = GEMINI_MODEL.replace("/", "_")

SOURCE = f"../data/gemini_harm_recheck_{MODEL_TAG}.csv"     # read only
CACHE  = f"../data/toxic_evidence/gemini_toxic_evidence_{MODEL_TAG}.jsonl" # NEW
OUT    = f"../data/toxic_evidence/gemini_toxic_evidence_{MODEL_TAG}.csv"   # NEW
os.makedirs("../data/toxic_evidence", exist_ok=True)

rc = pd.read_csv(SOURCE)
toxic = rc[rc["harm_mention_v2"] == "yes"].sample(frac=1, random_state=0).reset_index(drop=True)  # fixed shuffle
df = toxic.head(SAMPLE) if SAMPLE else toxic
print(f"{len(toxic):,} toxic reviews in total, this run covers {len(df):,}")
df["game"].value_counts()

1,669 toxic reviews in total, this run covers 1,669


game
Counter-Strike 2       843
PUBG: BATTLEGROUNDS    500
Grand Theft Auto V     326
Name: count, dtype: int64

## 3. Run

Every review is saved the moment it finishes. A second run skips what is saved.

In [25]:
with open("../prompts/toxic_evidence.txt", "r") as f:
    SYSTEM_EV = f.read()

run = toxic_evidence.EvidenceRun(df, client, GEMINI_MODEL, SYSTEM_EV, CACHE, temperature=TEMPERATURE, workers=8)
run.start()

1,669 reviews in scope · 49 already done (skipped) · 1,620 to run now
25/1,620 · 5.3 rev/s · ETA 4m58s · retries 0 · failed 2
50/1,620 · 6.5 rev/s · ETA 3m59s · retries 0 · failed 2
75/1,620 · 7.3 rev/s · ETA 3m32s · retries 0 · failed 3
100/1,620 · 7.5 rev/s · ETA 3m22s · retries 0 · failed 3
125/1,620 · 7.7 rev/s · ETA 3m13s · retries 0 · failed 3
150/1,620 · 7.9 rev/s · ETA 3m05s · retries 0 · failed 3
175/1,620 · 8.1 rev/s · ETA 2m58s · retries 0 · failed 3
200/1,620 · 8.1 rev/s · ETA 2m56s · retries 0 · failed 3
225/1,620 · 8.2 rev/s · ETA 2m49s · retries 0 · failed 3
250/1,620 · 8.3 rev/s · ETA 2m44s · retries 0 · failed 3
275/1,620 · 8.4 rev/s · ETA 2m40s · retries 0 · failed 3
300/1,620 · 8.4 rev/s · ETA 2m36s · retries 0 · failed 3
325/1,620 · 8.5 rev/s · ETA 2m32s · retries 0 · failed 3
350/1,620 · 8.5 rev/s · ETA 2m29s · retries 0 · failed 3
375/1,620 · 8.5 rev/s · ETA 2m26s · retries 0 · failed 3
400/1,620 · 8.5 rev/s · ETA 2m23s · retries 0 · failed 3
425/1,620 · 8.5 rev/s

## 4. Save the evidence table (new file) and check the quotes

In [26]:
ev = run.export(OUT)

ok_rate = ev["quote_in_review"].mean()
print(f"\nQuotes found word for word in the review: {ev['quote_in_review'].sum():,} of {len(ev):,} ({ok_rate:.1%})")
print("The rest are kept in the file but left out of the counts below.")
ev = ev[ev["quote_in_review"]].copy()
ev.head(10)[["game", "quote", "meaning_en", "type", "identity_group", "target"]]

Saved 2,541 pieces of evidence from 1,662 reviews -> ../data/toxic_evidence/gemini_toxic_evidence_gemini-3.5-flash-lite.csv  (42 reviews returned no evidence)

Quotes found word for word in the review: 2,445 of 2,541 (96.2%)
The rest are kept in the file but left out of the counts below.


,game,quote,meaning_en,type,identity_group,target
0,Grand Theft Auto V,Lester Çeneni Sikiyim,Fuck your jaw Lester,profanity,none,other
1,PUBG: BATTLEGROUNDS,f u c king,profanity directed at the game,profanity,none,developer
2,Counter-Strike 2,argentinos desprezíveis,despicable Argentinians,identity_attack,ethnicity_nationality,nationality_group
3,PUBG: BATTLEGROUNDS,韩国棒子,Korean stick (derogatory ethnic slur),identity_attack,ethnicity_nationality,nationality_group
4,PUBG: BATTLEGROUNDS,高丽的棒子,Goryeo stick (derogatory ethnic slur),identity_attack,ethnicity_nationality,nationality_group
5,PUBG: BATTLEGROUNDS,这个国家女的自己扣都比你们国家男的插得爽,Vulgar sexual insult directed at the nationality's citizens,sexual_harassment,none,nationality_group
6,PUBG: BATTLEGROUNDS,吔屎啦蓝洞,eat shit Bluehole,profanity,none,developer
7,PUBG: BATTLEGROUNDS,傻逼蓝洞,stupid cunt Bluehole,personal_insult,none,developer
8,PUBG: BATTLEGROUNDS,我玩你妈嗨,what the fuck am I playing,profanity,none,developer
9,Counter-Strike 2,salauds moqueurs,mocking bastards,personal_insult,none,player_general


## 5. Summary

First a short written summary, then one table: for each type of toxicity, how many toxic reviews have it, the share per game, the most common phrases, and a couple of readable examples.

In [27]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 250)

L = toxic_evidence
ev["Type"] = ev["type"].map(L.TYPE_LABELS)
ev["Aimed at"] = ev["target"].map(L.TARGET_LABELS)
ev["Group"] = ev["identity_group"].map(L.GROUP_LABELS)
ev = ev.merge(df[["key", "language"]].rename(columns={"language": "lang2"}), on="key", how="left")
ev["language"] = ev["language"].fillna(ev["lang2"])

n_toxic = df["key"].nunique()
n_with = ev["key"].nunique()
per_type = ev.drop_duplicates(["key", "type"]).groupby("type")["key"].nunique().sort_values(ascending=False)
top = per_type.index[0]
per_target = ev.drop_duplicates(["key", "target"]).groupby("target")["key"].nunique().sort_values(ascending=False)

print(f"{n_toxic:,} toxic reviews were checked. {n_with:,} ({n_with / n_toxic:.0%}) have at least one quoted phrase that was found in the review.")
print(f"Most common type: {L.TYPE_LABELS[top]}, in {per_type.iloc[0] / n_toxic:.0%} of toxic reviews.")
print("Order of types: " + ", ".join(f"{L.TYPE_LABELS[t].split(' (')[0]} {v / n_toxic:.0%}" for t, v in per_type.items()))
print(f"Most often aimed at: {L.TARGET_LABELS[per_target.index[0]]} ({per_target.iloc[0] / n_toxic:.0%} of toxic reviews), then {L.TARGET_LABELS[per_target.index[1]]} ({per_target.iloc[1] / n_toxic:.0%}).")
print("A review can have several types, so the shares add up to more than 100%.")

1,669 toxic reviews were checked. 1,570 (94%) have at least one quoted phrase that was found in the review.
Most common type: Insult at a person (skill, brain, character), in 49% of toxic reviews.
Order of types: Insult at a person 49%, Attack on who someone is 26%, Swearing, not aimed at anyone 20%, Toxic community in general 11%, Other 8%, Threat or wishing harm 6%, Sexual harassment 1%, Griefing or trolling 1%
Most often aimed at: Players in general (58% of toxic reviews), then A nationality or group (22%).
A review can have several types, so the shares add up to more than 100%.


In [28]:
rows = []
n_game = df.groupby("game")["key"].nunique()
for t in per_type.index:
    s = ev[ev["type"] == t]
    byg = s.drop_duplicates("key").groupby("game")["key"].nunique()
    rows.append({
        "Type": L.TYPE_LABELS[t],
        "Toxic reviews": s["key"].nunique(),
        "% of toxic reviews": round(s["key"].nunique() / n_toxic * 100, 1),
        "CS2 %": round(byg.get("Counter-Strike 2", 0) / n_game.get("Counter-Strike 2", 1) * 100, 1),
        "GTA V %": round(byg.get("Grand Theft Auto V", 0) / n_game.get("Grand Theft Auto V", 1) * 100, 1),
        "PUBG %": round(byg.get("PUBG: BATTLEGROUNDS", 0) / n_game.get("PUBG: BATTLEGROUNDS", 1) * 100, 1),
        "Common phrases (English meaning)": ", ".join(s["meaning_en"].str.lower().value_counts().head(4).index),
        "Examples": L.examples(s, n=2),
    })
summary = pd.DataFrame(rows)
summary

,Type,Toxic reviews,% of toxic reviews,CS2 %,GTA V %,PUBG %,Common phrases (English meaning),Examples
0,"Insult at a person (skill, brain, character)",810,48.5,43.5,55.2,52.6,"mentally disabled players, i am a stupid cunt, i am a fucking idiot, i am a stupid cunt / idiot","""thirsty lv500 neckbeards"" | ""Noobs"""
1,"Attack on who someone is (nationality, gender, etc.)",432,25.9,30.4,14.1,26.0,"derogatory term for russians, derogatory term for chinese people, expressing hatred towards specific nationalities, chinese people","""turned me racist"" | ""Russians"""
2,"Swearing, not aimed at anyone",336,20.1,16.6,25.2,22.8,"fuck your mother, sons of bitches, your mother's cunt, fuck","""Fuck you Rockstar. Fuck. You."" | ""suka blqt, idi nahui"""
3,Toxic community in general,186,11.1,17.4,9.8,1.4,"toxic community, toxic players, the community is toxic, the player base is extremely toxic","""the community is toxic"" | ""extremely racist and homophobic vocabulary"""
4,Other,130,7.8,10.2,5.5,5.2,"claims to have had relations with my mother, wish you go bankrupt soon, anyone who has thought about suicide, russian warship, go fuck yourself!","""Cancer"" | ""EU servers are fucked"""
5,Threat or wishing harm,103,6.2,5.1,6.1,8.0,"wish your mother dies first, wishing death upon the cheater's entire family, cheaters go die with your whole family, having neither father nor mother, wishing death upon the developer's family","""You are recommended to drink bleach"" | ""cease to exist"""
6,Sexual harassment,16,1.0,1.4,0.0,0.8,"vulgar sexual insult directed at the nationality's citizens, harassing, sent an unsolicited sexually explicit image, hookup chat messages","""jizz"" | ""sent me a picture of his private parts"""
7,Griefing or trolling,14,0.8,0.8,1.5,0.4,"to annoy teammates to the absolute maximum, personal private information was exposed in public chat, someone who intentionally disrupts the gameplay of others, cheaters are overjoyed","""trolls, greifers"" | ""teammate killing other teammates"""


## 6. Who is it aimed at?

In [29]:
rows = []
for t in per_target.index:
    s = ev[ev["target"] == t]
    rows.append({"Aimed at": L.TARGET_LABELS[t], "Toxic reviews": s["key"].nunique(),
                 "% of toxic reviews": round(s["key"].nunique() / n_toxic * 100, 1), "Examples": L.examples(s, n=2)})
pd.DataFrame(rows)

,Aimed at,Toxic reviews,% of toxic reviews,Examples
0,Players in general,975,58.4,"""Some of you are morons here"" | ""encourages racism"""
1,A nationality or group,363,21.7,"""Chinese player nonsense"" | ""more Russians"""
2,The developer,284,17.0,"""pos game"" | ""shit ass game"""
3,Teammates,124,7.4,"""С*ка Бли@т"" | ""fu**ink NOOB!"""
4,Other,58,3.5,"""Cancer"" | ""suka blqt, idi nahui"""
5,Opponents,17,1.0,"""cuck"" | ""Не позорьте мать!"" (Do not bring shame to your mother!)"


## 7. Attacks on identity: which group?

Only the evidence typed as an attack on who someone is.

In [30]:
ia = ev[ev["type"] == "identity_attack"]
print(f"{ia['key'].nunique():,} toxic reviews ({ia['key'].nunique() / n_toxic:.0%}) have an attack on identity.\n")
rows = []
for g, s in ia.groupby("identity_group"):
    rows.append({"Group": L.GROUP_LABELS[g], "Toxic reviews": s["key"].nunique(),
                 "% of identity attacks": round(s["key"].nunique() / ia["key"].nunique() * 100, 1),
                 "Examples": L.examples(s, n=2)})
pd.DataFrame(rows).sort_values("Toxic reviews", ascending=False).reset_index(drop=True)

432 toxic reviews (26%) have an attack on identity.



,Group,Toxic reviews,% of identity attacks,Examples
0,Ethnicity or nationality,379,87.7,"""CHINA CHEATERS"" | ""racial profanity"""
1,Disability,18,4.2,"""rеtаrdеd individuals"" | ""autism"""
2,Sexuality,16,3.7,"""kick etogo pidara"" | ""faggot"""
3,Other group,12,2.8,"""turned me racist"" | ""made me a little bit racist"""
4,Gender,11,2.5,"""women have no rights"" | ""make fun of my voice and ask me if I'm a girl"""
5,Religion,2,0.5,"""made fun of for being a jew"" | ""Allahu akbar simulator"" (referencing Islamic phrase mockingly as a simulator)"


## 8. By game

Number of toxic reviews with an attack on identity, per game.

In [31]:
ia.drop_duplicates("key").groupby("game").size().to_frame("Reviews with an attack on identity").join(n_game.rename("Toxic reviews"))

,Reviews with an attack on identity,Toxic reviews
game,,
Counter-Strike 2,256,843
Grand Theft Auto V,46,326
PUBG: BATTLEGROUNDS,130,500


## 9. Most common phrases per type

The top meanings in English, so phrases in different languages are counted together.

In [32]:
for t in per_type.index:
    s = ev[ev["type"] == t]
    print(f"\n=== {L.TYPE_LABELS[t]}: {s['key'].nunique():,} reviews ===")
    print(s["meaning_en"].str.lower().value_counts().head(8).to_string())


=== Insult at a person (skill, brain, character): 810 reviews ===
meaning_en
mentally disabled players          29
i am a stupid cunt                 20
i am a fucking idiot               17
i am a stupid cunt / idiot         14
calling a russian player a noob    13
cheaters have no mother            12
i am an idiot                       7
trash game                          7

=== Attack on who someone is (nationality, gender, etc.): 432 reviews ===
meaning_en
derogatory term for russians                        13
derogatory term for chinese people                   8
expressing hatred towards specific nationalities     5
chinese people                                       5
tell a russian player to shut the fuck up            5
russians                                             4
shut the fuck up russian                             4
russian children                                     3

=== Swearing, not aimed at anyone: 336 reviews ===
meaning_en
fuck your mother      24
sons

## 10. Human spot check

Gemini explains its own labels, so check a few by hand. This writes 30 random pieces of evidence to a new CSV. Fill `human_ok` with `yes` or `no` (is the type right, and does the quote really show toxicity?). The share of `yes` tells us how far to trust the counts above.

In [33]:
check = ev.sample(min(30, len(ev)), random_state=7)[["key", "game", "quote", "meaning_en", "Type", "Group", "Aimed at", "review"]].copy()
check["human_ok"] = ""
CHECK_OUT = f"../data/toxic_evidence/toxic_evidence_spotcheck_{MODEL_TAG}.csv"
check.to_csv(CHECK_OUT, index=False)
print(f"Saved {len(check)} rows -> {CHECK_OUT}")
check[["quote", "meaning_en", "Type", "Aimed at"]].head(10)

Saved 30 rows -> ../data/toxic_evidence/toxic_evidence_spotcheck_gemini-3.5-flash-lite.csv


,quote,meaning_en,Type,Aimed at
2261,중국 Die,China die,"Attack on who someone is (nationality, gender, etc.)",A nationality or group
1588,RUSLAR KAPATILSIN,Close down the Russians,"Attack on who someone is (nationality, gender, etc.)",A nationality or group
751,Russian Noob,Russian novice or bad player,"Attack on who someone is (nationality, gender, etc.)",A nationality or group
1126,росія лох,Russia is a loser,"Attack on who someone is (nationality, gender, etc.)",A nationality or group
423,wdnmd,Profane insult acronym (your mother's dead),"Swearing, not aimed at anyone",The developer
1454,bacılarını yerler umarım,I hope they violate their sisters,Other,Players in general
341,垃圾游戏,garbage game,Other,The developer
780,"Eindeutige Kaufempfehlung für jeden, der an Suizid dachte und sich nicht vor die Bahn schmeißen will.",Recommendation for anyone who thought about suicide and does not want to jump in front of a train,Other,Players in general
2324,操！你！！妈,fuck your mother,"Swearing, not aimed at anyone",The developer
1312,tous les Kevin de la planète,all the annoying childish players,"Insult at a person (skill, brain, character)",Players in general
